In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week9-lesson-4"). \
config("spark.sql.warehouse.dir", f"/user/itv027484/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
spark

In [3]:
order_schema= 'order_id long, order_date string , customer_id long, order_status string'

In [4]:
orders_df = spark.read.format('csv').schema(order_schema).load('/public/trendytech/orders/orders_1gb.csv')

In [5]:
orders_df.show(4)

+--------+--------------------+-----------+---------------+
|order_id|          order_date|customer_id|   order_status|
+--------+--------------------+-----------+---------------+
|       1|2013-07-25 00:00:...|      11599|         CLOSED|
|       2|2013-07-25 00:00:...|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|      12111|       COMPLETE|
|       4|2013-07-25 00:00:...|       8827|         CLOSED|
+--------+--------------------+-----------+---------------+
only showing top 4 rows



### Currently only 2 executors allocated, and hence parallelism is just 2

In [6]:
spark.sparkContext.defaultParallelism

2

In [7]:
orders_df.count()

25831125

### For executing count(), additional executors were added and then the parallelism increased 

In [8]:
spark.sparkContext.defaultParallelism

3

#### DF has 9 partitions. 1gb+  ~ 128MB per partition

In [9]:
orders_df.rdd.getNumPartitions()

9

In [10]:
orders_df.createOrReplaceTempView("orders")

In [11]:
spark.sql("select count(distinct order_status) from orders").show()

+----------------------------+
|count(DISTINCT order_status)|
+----------------------------+
|                           9|
+----------------------------+



### total 210 tasks for count distinct. 9 for initial step , then 200 for shuffle/sort finally 1 for count

In [12]:
spark.sparkContext.defaultParallelism

7

In [13]:
spark.sql("select count(*) from orders where order_status = 'CLOSED'").show()

+--------+
|count(1)|
+--------+
| 2833500|
+--------+

